In [1]:
# numpy(넘파이) 는 숫자 여러개를 '배열(array)'로 묶어 한 번에 계산하게 해 주는 라이브러리 입니다.
# 예) [160, 170, 180]에 모두 2를 더하는 계산을 한 줄로 할 수 있습니다.
# 이 노트북에서는 데이터 준비/정규화/학습 전 확인 같은 'NumPy 흐름' 에 사용합니다.

import numpy as np

# torch 는 PyTorch 라이브러리 입니다. 
# 이번 노트북에서는 (1) 자동 미분(autograd)과 (2) optimizer(torch.optim.SGD)를 쓰기 위해 사용합니다.
#  - 자동 미분 : 사람이 미분 공식을 적지 않아도 PyTorch 가 a.grad, b.grad를 계산해 주는 기능
#  - optimizer : 우리가 직접 하던 a, b 업데이트를 PyTorch 가 대신 수행해 주는 도구
import torch

from math import e 

In [2]:
#=================================================================
# 1. 입력값 X와 y의 준비 (이전 파일과 동일)
#=================================================================

# X: 입력값입니다. 여기서는 사람의 키(cm)를 사용합니다.
# np.array([...])는 여러 숫자를 하나의 NumPy 배열로 묶는 것입니다.
X = np.array([160, 170, 180, 190])

# 정답 값입니다. 0은 농구선수 아님, 1은 농구선수 입니다.
# X 와 y 는 순서대로 짝지어집니다.  즉 키 160 → 정답 0 , 키 190 → 정답 1.
y = np.array([0, 0, 1, 1])

print("입력값 X:", X)
print("정답값 y:", y)


입력값 X: [160 170 180 190]
정답값 y: [0 0 1 1]


In [3]:
#=================================================================
# 2. 입력값 정규화
#=================================================================

# 평균(mean)과 표준편차(std)를 계산합니다.
# - 평균 : 데이터의 중심(가운데 쯤 되는 값)
# - 표준편차 : 데이터가 평균에서 얼마나 넓게 퍼져 있는지 나타내는 값
X_mean = np.mean(X)
X_std = np.std(X)

# 정규화 공식: (원본값 - 평균) / 표준편차
# 주의: 실제 학습에는 원래 키 X가 아니라, 정규화 된 입력값 X_norm 을 사용합니다.
#         (X_mean, X_std 는 나중에 '새 입력값 예측' 에서도 똑같이 다시 씁니다.)
X_norm = (X - X_mean) / X_std

print("입력값 평균 X_mean:", X_mean)
print("입력값 표준편차 X_std:", X_std)
print("정규화된 입력값 X_norm:", X_norm)

입력값 평균 X_mean: 175.0
입력값 표준편차 X_std: 11.180339887498949
정규화된 입력값 X_norm: [-1.34164079 -0.4472136   0.4472136   1.34164079]


In [4]:
#=================================================================
# 2-1. X_norm 과  y 를 PyTorch tensor 로 변환 (이전 파일과 동일)
#=================================================================

# 학습에 사용할 입력값(X_norm)과 정답(y)을 tensor로 바꿔 둡니다.

# 주의 (햇갈리기 쉬움):
#     X      = 원래 키(cm)
#     X_norm = 정규화 된 입력값 ← 학습에는 이 값을 사용합니다.
# 따라서 아래에서도 X 가 아니라 X_norm 을 tensor로 변환합니다.

# dtype = torch.float32: 소수점 계산(미분) 을 위해 실수(float) 형식으로 만듭니다.float

X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
y_tensor = torch.tensor(y, dtype = torch.float32)

print("학습용 입력 tensor X_norm_tensor:", X_norm_tensor)
print("학습용 정답 tensor y_tensor:", y_tensor)


학습용 입력 tensor X_norm_tensor: tensor([-1.3416, -0.4472,  0.4472,  1.3416])
학습용 정답 tensor y_tensor: tensor([0., 0., 1., 1.])


In [5]:
#=================================================================
# 3. 가중치 a와 편향 b 초기값 설정 (requires_grad = True 인 tensor)
#=================================================================

# a 는 가중치(weight), b 는 편향(bias)입니다.
# a 는 원래 키(cm) 가 아니라 정규화된 입력값 X_norm 에 곱해지는 값 입니다.

# requires_grad = True 는 "이 값에 대한 미분값(기울기)을 계산해 줘" 라는 설정입니다.
# a, b 는 학습으로 계속 수정해 나갈 값이므로 requires_grad = True 를 설정합니다.
# (입력 X_norm_tensor, 정답 y_tensor 에는 이 설정을 하지 않았습니다.
# 그 둘은 학습으로 바꾸는 값이 아니기 때문입니다.)
a = torch.tensor(0.1, dtype = torch.float32, requires_grad = True)
b = torch.tensor(0.1, dtype = torch.float32, requires_grad = True)

# a,b 는 tensor라서 그냥 출력하면 tensor (...) 형태로 나옵니다.
# .item()으로 숫자만 깔끔하게 꺼내서 출력합니다.
print("초기 가중치 a : ",a.item())
print("초기 가중치 b : ",b.item())

초기 가중치 a :  0.10000000149011612
초기 가중치 b :  0.10000000149011612


In [6]:
#=================================================================
# 4. 학습 설정 (learning_rate, epochs)
# #=================================================================

# learning_rate(학습률): 한번에 a, b를 얼마나 크게 수정할지 정하는 값입니다.
# 이 값은 바로 다음 셀에서 optimizer를 만들때 lr = learning_rate 로 넘겨줍니다.
learning_rate = 0.1

# epochs(에폭): 전체 데이터를 몇 번 반복해서 학습할 지 정하는 값입니다.
# 여기서는 같은 데이터롤 1000번 반복 학습합니다.
epochs = 1000

print("learning_rate:", learning_rate)
print("epochs:", epochs)

learning_rate: 0.1
epochs: 1000


In [7]:
#=================================================================
# 5. optimizer 생성(이번 실습의 핵심)
#=================================================================

torch.optim.SGD([a,b], learning_rate)
#    - [a, b]         : optimizer가 업데이트 할 학습 대상(파라미터) 입니다.
#                       optimizer는 a.grad, b.grad를 이용해 a, b를 업데이트 합니다.
#    - lr - learning_rate : 한 번에 얼마다 움직일 지(학습률). 위에서 정한 0.1 을 넘겨줍니다.
   
   
#    이 optimizer 가 학습 루프에서 하는 일은 단 두가지 입니다.
#    optimizer.zero_grad(): a.grad, b.grad를 0 으로 초기화(이전의 a.grad.zero_(), b.grad.zero_() 역할)
#    optimizer.step(): a, b를 업데이트(이전의 a -= lr*a.grad, b -= lr*b.grad 역할)
optimizer = torch.optim.SGD([a,b], lr = learning_rate)
   
print("Optimizer 준비 완료:", optimizer)

Optimizer 준비 완료: SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)


In [8]:
#=================================================================
# 6. 시그모이드 함수 정의 (학습 전 NumPy 확인용)
#=================================================================

# 시그모이드 함수는 계산 값 H(x)를 0~1 사이의 예측 확률 z로 변환합니다.
# 수식: sigmoid(H) = 1 / (1 + e**(-H))
#         - H 가 크면(양수로 클수록) 결과가 1에 가까워지고,
#         - H 가 작으면(음수로 작을수록) 결과가 0에 가까워집니다.
        
# 아래 함수는 '학습 전 예측 / Cost 확인(NumPy 흐름)에서만 사용합니다.'
# 실제 학습 루프 안에서는 PyTorch의 torch.sigmoid 를 사용합니다.
def sigmoid(H):
    return 1 / (1 + e**(-H))

In [9]:
#=================================================================
# 7. 학습 전 예측 확인 (이전 파일과 동일한 NumPy 흐름)
#=================================================================

# H(x) = a * X_norm + b (sigmoid 전의 선형 계산값 - 확률이 아닙니다.)
# 학습 전 상태를 눈으로 확인하려고 NumPy 로 잠깐 계산해 봅니다.
# a, b 는 tensor 이고 X_norm은 NumPy 배열이라, .item()으로 숫자만 꺼내 함께 계산합니다.
H = a.item() * X_norm + b.item()

# z = sigmoid(H) ← H(x)가 아니라 z가 확률입니다.

z = sigmoid(H)

print("학습 전 선형 계산값 H(x):", H)
print("학습 전 시그모이드 예측 확률 z:", z)

학습 전 선형 계산값 H(x): [-0.03416408  0.05527864  0.14472136  0.23416408]
학습 전 시그모이드 예측 확률 z: [0.49145981 0.51381614 0.53611732 0.55827498]


In [10]:
#=================================================================
# 8. 학습 전 비용(Cost) 계산 (이전 파일과 동일한 NumPy 흐름)
#=================================================================

# Binary Cross Entropy(BCE) 비용 함수입니다. (BCE = 이진 교차 엔트로피)
# 실제값 y 가 1이면 z가 1에 가까울수록 Cost가 작아지고
# 실제값 y 가 0이면 z가 0에 가까울수록 Cost가 작아집니다.
# Cost = - {y * log(z) + (1 - y) * log(1 - z)}

# log(0)을 방지하기 위해 z가 정확히 0 또는 1이 되지 않도록 제한합니다.
# NumPy 에서는 np.clip()을 사용합니다. 학습 루프에서는 torch.clamp()를 씁니다.
epsilon = 1e-7
z_safe = np.clip(z, epsilon, 1-epsilon)

costs = (y * np.log(z_safe) + (1 - y) * np.log(1 - z_safe))
Cost = np.mean(costs)

print("학습 전 각 샘플의 비용 (Cost):", costs)
print("학습 전 평균 비용 (Cost):", Cost)

학습 전 각 샘플의 비용 (Cost): [-0.67621103 -0.72116842 -0.62340225 -0.58290364]
학습 전 평균 비용 (Cost): -0.6509213354683427


In [14]:
#=================================================================
# 9. optimizer로 경사 하강법 학습 (이번 실습의 핵심 루프)
#=================================================================

# 한번의 epoch 에서 일어나는 단계 (반드시 이 순서):
#     1. optimizer.zero_grad() : 이전 epoch 의 grad 를 0으로 초기화
#     2. H = a * X_norm + b    : 선형 계산값
#     3. z = sigmoid(H)        : 예측 확률
#     4. z_safe = clamp(z)     : log(0) 방지를 위해 z 범위 제한
#     5. Cost = BCE(z_safe, y) : 비용 계산
#     6. Cost.backward()       : a.grad, b.grad 자동 계산
#     7. optimizer.step()      : a, b 업데이트
#     8. 학습상태 출력

for epoch in range(epochs):
    
    # ----- 1. 이전 epoch 에서 계산된 grad 초기화 ----
    # PyTorch 의 grad는 덮어쓰기가 아니라 '누적(더하기)' 됩니다.
    # 이전 epoch의 a.grad, b.grad 가 남아있으면 이번 grad 와 더해져 잘못된 업데이트가 됩니다.
    # 그래서 forward 계산과 Cost.backward() '전에' 먼저 초기화 합니다.
    # optimizer는 [a,b] 를 관리하므로, 이 한줄이 a.grad.zero_(), b.grad.zero_() 를 대신합니다.
    optimizer.zero_grad()
    
    # ---- 2. H(x) = a*X_norm + b 계산 ----
    # H(x) 는 sigmoid 에 들어가기 전의 선형 계산값 입니다. (확률이 아닙니다.)
    # 학습용이므로 tensor 인 a, b 와 X_norm_tensor 를 그대로 곱합니다. (.item()을 쓰지않습니다.)
    H = a * X_norm_tensor + b
    
    # ---- 3. sigmoid를 적용해 예측 확률 z 계산 ----
    # z 는 0 ~ 1 사이의 예측 확률입니다.
    z = torch.sigmoid(H)
    
    # ---- 4. BCE Cost 계산에서 log(0) 이 발생하지 않도록 z 범위 제한 ----
    # torch.clamp()는 NumPy 의 np.clip() 과 같은 역할(값을 범위 안으로 제한) 을 합니다.
    z_safe = torch.clamp(z, epsilon, 1-epsilon)
    
    # ---- 5. Binary Cross Entropy Cost 계산 ----
    # Cost = - {y * log(z) + (1 - y) * log(1 - z)} 를 4개 데이터에 대해 평균 냅니다.
    costs = -(
            y_tensor * torch.log(z_safe)
            + (1 - y_tensor) * torch.log(1 - z_safe)
    )
    mean_cost = torch.mean(costs)
    
    # ----6. backward: a.grad, b.grad 자동 계산 ----
    # mean_cost 에서 출발해 a,b 까지 거꾸로 따라가며 미분값을 구해
    #     a.grad = grad_a 평균((z - y) * X_norm )
    #     b.grad = grad_b 평균(z - y)
    # 에 저장합니다.
    mean_cost.backward()
    
    # ----7. optimizer 가 a와 b 업데이트 ----
    # optimizer.step()은 a.grad, b.grad 를 사용해 Cost 가 줄어드는 방향으로 a, b 를 수정합니다.
    #   a = a - learning_rate * grad_a
    #   b = b - learning_rate * grad_a
    #   (이전 노트북의 'with torch.no_grad(): a -= ...' 블록을 이 한 줄이 대신합니다.)
    optimizer.step()
    
    # ----8. 학습 상태 출력 ----
    # tensor가 그대로 출력되지 않도록 .item()으로 숫자만 꺼내 출력 합니다.
    # 100 epoch 마다 한번씩, 그리고 마지막 epoch 에서 출력합니다.
    if epoch % 100 == 0 or epoch == epochs -1:
        print(
            f"epoch = {epoch}",
            f"Cost = {mean_cost.item():.6f}",
            f"a = {a.item():.6f}",
            f"b = {b.item():.6f}"
        )
    
    # (참고) 초반 3 epoch 에서만 a.grad, b.grad 값을 확인해 봅니다.
    # optimizer.step() 뒤에도 grad는 다음 epoch 의 zero_grad() 전까지 그대로 남아있어 확인할 수 있습니다.
    # grad_a, grad_b 는 각각 a.grad, b.grad 와 같은 의미입니다.
    if epoch < 3:
        grad_a = a.grad.item()   
        grad_b = b.grad.item()   
        print(
            f"(확인용) a.grad = {grad_a:.6f}",
            f"b.grad = {grad_b:.6f}"
        )

epoch = 0 Cost = 0.650921 a = 0.142231 b = 0.097508
(확인용) a.grad = -0.422310 b.grad = 0.024917
(확인용) a.grad = -0.411837 b.grad = 0.024236
(확인용) a.grad = -0.401671 b.grad = 0.023556
epoch = 100 Cost = 0.198213 a = 2.069432 b = 0.016209
epoch = 200 Cost = 0.133778 a = 2.860825 b = 0.005363
epoch = 300 Cost = 0.104190 a = 3.401657 b = 0.002251
epoch = 400 Cost = 0.086188 a = 3.824506 b = 0.001083
epoch = 500 Cost = 0.073783 a = 4.175877 b = 0.000572
epoch = 600 Cost = 0.064611 a = 4.478189 b = 0.000324
epoch = 700 Cost = 0.057510 a = 4.744266 b = 0.000194
epoch = 800 Cost = 0.051832 a = 4.982254 b = 0.000122
epoch = 900 Cost = 0.047180 a = 5.197718 b = 0.000079
epoch = 999 Cost = 0.043329 a = 5.392772 b = 0.000054


In [16]:
#=================================================================
# 10. 학습 완료 후 최종 가중치와 편향 확인
#=================================================================

# 학습된 a와 b는 optimizer.step()에 의해 1000번 반복 업데이트된 값입니다.
# (정규화된 입력값 X_norm을 기준으로 학습된 값이라는 점을 기억하세요.)
print("학습된 a:", a.item())
print("학습된 b:", b.item())

학습된 a: 5.3927717208862305
학습된 b: 5.3598680096911266e-05


In [17]:
#=================================================================
# 11. 새로운 입력값 예측
#=================================================================

# 키가 185cm 인 사람이 농구선수인지 예측합니다.
input_height = 185

# 새로운 입력값에도 학습 데이터와 '같은 기준' 으로 정규화 해야 합니다.
# 학습 때 사용한 X_mean, X_std 를 그대로 다시 사용합니다. (새로 계산하면 안됩니다.)
input_norm = (input_height - X_mean) / X_std

# # 예측은 학습이 아니므로 a,b 를 업데이트 하지 않습니다.
# 따라서 미분 계산 기록도 필요 없으므로 with torch.no_grad() 안에서 계산합니다.
with torch.no_grad():
    # 학습된 a, b는 tensor 이므로, 입력 값도 tensor로 맞춰서 계산합니다.
    input_norm_tensor = torch.tensor(input_norm, dtype = torch.float32)
    # H(x) = a*X_norm + b (선형 계산값 - 확률이 아닙니다.)
    H_new = a * input_norm_tensor + b
    # z = sigmoid(H_new)
    z_new = torch.sigmoid(H_new)
    # 0.5 이상이면  1(농구선수), 미만이면 0 (농구선수 아님)
    pred = 1 if z_new.item() >= 0.5 else 0
    
print(f"키가 {input_height}cm 인 사람의 농구선수 확률(z): {z_new.item():.4f}")
if pred == 1:
    print("판별 결과: 농구선수 입니다.")
else:
    print("판별 결과: 농구선수가 아닙니다.")

키가 185cm 인 사람의 농구선수 확률(z): 0.9920
판별 결과: 농구선수 입니다.
